## Problem and data description

### Decision problem

A fleet of identical vehicles must leave one depot, serve every customer exactly once, and return to the depot. Each customer has a non-negative demand, and the total demand assigned to a vehicle cannot exceed its capacity. The objective is to minimize total travel distance.

This is representative of last-mile delivery, collection, maintenance, and bike-rebalancing operations. A good solution lowers distance, vehicle use, fuel consumption, and operating time while satisfying all service and capacity requirements.

In [7]:
import json
import math
import os
import platform
import random
import time
from datetime import datetime, timezone
from pathlib import Path

import folium
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pulp
import psutil
import requests
from IPython.display import FileLink, display
from scipy.spatial.distance import cdist

# Use one baseline seed so the primary synthetic experiment is reproducible.
SEED = 42
np.random.seed(SEED)
random.seed(SEED)
pd.set_option("display.max_columns", None)

# Record the execution environment because hardware and solver versions affect runtime comparisons.
print("Python experiment environment loaded successfully.")
runtime_environment = pd.DataFrame(
    {
        "Component": [
            "Python", "Operating system", "Machine", "Processor",
            "Logical CPU cores", "Memory (GB)", "PuLP", "HiGHS",
            "Folium", "Timing boundary",
        ],
        "Value": [
            platform.python_version(),
            platform.platform(),
            platform.machine(),
            platform.processor() or os.environ.get("PROCESSOR_IDENTIFIER", "Not reported"),
            psutil.cpu_count(logical=True),
            round(psutil.virtual_memory().total / (1024 ** 3), 2),
            pulp.__version__,
            getattr(__import__("highspy"), "__version__", "Available through PuLP"),
            folium.__version__,
            "Includes model/population construction and optimization, but excludes data generation and plotting",
        ],
    }
)
display(runtime_environment)


Python experiment environment loaded successfully.


,Component,Value
0,Python,3.13.2
1,Operating system,Windows-11-10.0.26300-SP0
2,Machine,AMD64
3,Processor,"Intel64 Family 6 Model 183 Stepping 1, Genuine..."
4,Logical CPU cores,28
5,Memory (GB),31.75
6,PuLP,4.0.0
7,HiGHS,Available through PuLP
8,Folium,0.20.0
9,Timing boundary,Includes model/population construction and opt...


### Controlled synthetic benchmark

The controlled benchmark places a depot at the centre of a 100 by 100 service region and generates customers with demands between 1 and 10 units. One coordinate unit is treated as one abstract distance unit. Euclidean distance represents travel in an unobstructed or approximately grid-like region. This abstraction is useful for controlled algorithm comparison but does not reproduce one-way streets, traffic, or road curvature.

Separate seeded random streams generate coordinates and demands, so the experiments are reproducible and smaller instances are prefixes of larger instances generated with the same seed. Synthetic data is used here to control problem size and isolate scalability effects. A later section uses a saved real Citi Bike snapshot and road-distance results to provide real-world validation.

In [8]:
# Validate every instance before optimization so invalid inputs cannot produce misleading results.
def validate_instance_data(instance):
    required = {
        "num_nodes", "num_customers", "num_vehicles",
        "vehicle_capacity", "dist_matrix", "demands",
    }
    missing = sorted(required.difference(instance))
    if missing:
        raise ValueError(f"Missing instance fields: {missing}")

    n = int(instance["num_customers"])
    node_count = int(instance["num_nodes"])
    vehicle_count = int(instance["num_vehicles"])
    capacity = float(instance["vehicle_capacity"])
    demands = np.asarray(instance["demands"], dtype=float)
    distances = np.asarray(instance["dist_matrix"], dtype=float)

    if n < 2 or node_count != n + 1:
        raise ValueError("An instance must contain a depot and at least two customers.")
    if vehicle_count <= 0 or capacity <= 0:
        raise ValueError("Vehicle count and capacity must be positive.")
    if demands.shape != (node_count,) or distances.shape != (node_count, node_count):
        raise ValueError("Demand vector or distance-matrix dimensions do not match the node count.")
    if not np.all(np.isfinite(demands)) or np.any(demands < 0) or demands[0] != 0:
        raise ValueError("Demands must be finite and non-negative, with zero demand at the depot.")
    if np.any(demands[1:] > capacity):
        raise ValueError("At least one customer demand exceeds vehicle capacity.")
    if demands.sum() > vehicle_count * capacity:
        raise ValueError("Total demand exceeds total fleet capacity.")
    if not np.all(np.isfinite(distances)) or np.any(distances < 0):
        raise ValueError("Distances must be finite and non-negative.")
    if not np.allclose(np.diag(distances), 0):
        raise ValueError("The distance-matrix diagonal must be zero.")
    return True


# Generate a controlled CVRP instance with independent seeded streams for coordinates and demands.
def generate_vrp_instance(
    num_customers=15,
    grid_size=100,
    max_demand=10,
    vehicle_capacity=30,
    num_vehicles=4,
    seed=42,
):
    coordinate_rng = np.random.default_rng(seed)
    demand_rng = np.random.default_rng(seed + 1)

    # Fix the depot at the centre and sample customers throughout the service region.
    depot_coords = np.array([[grid_size / 2, grid_size / 2]], dtype=float)
    customer_coords = coordinate_rng.uniform(0, grid_size, size=(num_customers, 2))
    coords = np.vstack([depot_coords, customer_coords])
    dist_matrix = cdist(coords, coords, metric="euclidean")
    demands = np.r_[0, demand_rng.integers(1, max_demand + 1, size=num_customers)]

    data = {
        "num_nodes": num_customers + 1,
        "num_customers": num_customers,
        "num_vehicles": num_vehicles,
        "vehicle_capacity": vehicle_capacity,
        "coords": coords,
        "dist_matrix": dist_matrix,
        "demands": demands,
        "distance_unit": "abstract distance units",
        "seed": seed,
    }
    validate_instance_data(data)
    return data


data = generate_vrp_instance(
    num_customers=10,
    vehicle_capacity=25,
    num_vehicles=3,
    seed=SEED,
)

instance_summary = pd.DataFrame(
    {
        "Metric": ["Customers", "Vehicles", "Capacity per vehicle", "Total demand", "Total fleet capacity", "Seed"],
        "Value": [
            data["num_customers"],
            data["num_vehicles"],
            data["vehicle_capacity"],
            int(data["demands"].sum()),
            data["num_vehicles"] * data["vehicle_capacity"],
            data["seed"],
        ],
    }
)
display(instance_summary)

# Present the generated inputs explicitly so the experiment can be audited and reproduced.
node_labels = ["Depot"] + [f"Customer_{customer}" for customer in range(1, data["num_customers"] + 1)]
synthetic_dataset = pd.DataFrame(
    {
        "Node_ID": range(data["num_nodes"]),
        "Node_Type": ["Depot"] + ["Customer"] * data["num_customers"],
        "X_coordinate": data["coords"][:, 0],
        "Y_coordinate": data["coords"][:, 1],
        "Demand": data["demands"].astype(int),
    },
    index=node_labels,
)
synthetic_dataset.index.name = "Node"
print("Complete controlled synthetic dataset")
display(synthetic_dataset.round({"X_coordinate": 3, "Y_coordinate": 3}))

distance_matrix_display = pd.DataFrame(
    data["dist_matrix"],
    index=node_labels,
    columns=node_labels,
).round(2)
distance_matrix_display.index.name = "From / To"
print(f"Complete Euclidean distance matrix ({data['distance_unit']})")
display(distance_matrix_display)


,Metric,Value
0,Customers,10
1,Vehicles,3
2,Capacity per vehicle,25
3,Total demand,49
4,Total fleet capacity,75
5,Seed,42


Complete controlled synthetic dataset


,Node_ID,Node_Type,X_coordinate,Y_coordinate,Demand
Node,,,,,
Depot,0,Depot,50.000,50.000,0
Customer_1,1,Customer,77.396,43.888,6
Customer_2,2,Customer,85.860,69.737,7
Customer_3,3,Customer,9.418,97.562,5
Customer_4,4,Customer,76.114,78.606,1
Customer_5,5,Customer,12.811,45.039,6
Customer_6,6,Customer,37.080,92.676,1
Customer_7,7,Customer,64.387,82.276,3
Customer_8,8,Customer,44.341,22.724,9


Complete Euclidean distance matrix (abstract distance units)


,Depot,Customer_1,Customer_2,Customer_3,Customer_4,Customer_5,Customer_6,Customer_7,Customer_8,Customer_9,Customer_10
From / To,,,,,,,,,,,
Depot,0.00,28.07,40.93,62.52,38.73,37.52,44.59,35.34,27.86,43.96,35.31
Customer_1,28.07,0.00,27.20,86.61,34.74,64.59,63.29,40.53,39.25,43.45,20.01
Customer_2,40.93,27.20,0.00,81.35,13.18,77.11,53.90,24.87,62.72,70.27,7.26
Customer_3,62.52,86.61,81.35,0.00,69.34,52.63,28.09,57.05,82.59,102.15,81.01
Customer_4,38.73,34.74,13.18,69.34,0.00,71.65,41.49,12.29,64.28,75.12,16.81
Customer_5,37.52,64.59,77.11,52.63,71.65,0.00,53.46,63.61,38.63,57.56,72.26
Customer_6,44.59,63.29,53.90,28.09,41.49,53.46,0.00,29.22,70.33,88.23,54.39
Customer_7,35.34,40.53,24.87,57.05,12.29,63.61,29.22,0.00,62.84,76.42,26.51
Customer_8,27.86,39.25,62.72,82.59,64.28,38.63,70.33,62.84,0.00,19.76,55.78


## Method 1 Integer Linear Programming

ILP is appropriate because CVRP decisions are discrete and its operational constraints can be written explicitly. When the solver reports `Optimal`, the returned objective is globally optimal for the model. The weakness is computational growth: CVRP is NP-hard, and the number of binary arc variables grows approximately with (O(mn^2)). A time-limited run may return a valid incumbent without proving optimality; therefore solver status, best bound, and MIP gap are reported rather than assuming every ILP result is optimal.


In [9]:
# Solve the capacitated VRP as a mixed-integer linear program using HiGHS.
def solve_vrp_exact_ilp(data, time_limit_sec=60):
    validate_instance_data(data)
    n = data["num_customers"]
    nodes = list(range(n + 1))
    customers = list(range(1, n + 1))
    vehicles = list(range(data["num_vehicles"]))
    costs = data["dist_matrix"]
    demands = [float(value) for value in data["demands"]]
    capacity = float(data["vehicle_capacity"])

    start_time = time.perf_counter()
    problem = pulp.LpProblem("Capacitated_VRP", pulp.LpMinimize)
    # x[i,j,k] equals 1 when vehicle k travels directly from node i to node j.
    x = problem.add_variable_dicts(
        "x",
        ((i, j, k) for i in nodes for j in nodes for k in vehicles if i != j),
        cat="Binary",
    )
    # u variables support the load-based MTZ subtour-elimination constraints.
    u = problem.add_variable_dicts(
        "u",
        ((i, k) for i in customers for k in vehicles),
        lowBound=0,
        upBound=capacity,
        cat="Continuous",
    )

    # Minimize total distance across all selected vehicle arcs.
    problem += pulp.lpSum(
        costs[i][j] * x[i, j, k]
        for i in nodes for j in nodes if i != j for k in vehicles
    )

    # Every customer must have exactly one incoming visit across the fleet.
    for customer in customers:
        problem += pulp.lpSum(
            x[i, customer, k]
            for i in nodes if i != customer for k in vehicles
        ) == 1

    # Enforce vehicle flow conservation, depot usage, capacity, and subtour elimination.
    for k in vehicles:
        for node in nodes:
            problem += pulp.lpSum(x[i, node, k] for i in nodes if i != node) == pulp.lpSum(
                x[node, j, k] for j in nodes if j != node
            )
        problem += pulp.lpSum(x[0, j, k] for j in customers) <= 1
        problem += pulp.lpSum(
            demands[i] * x[i, j, k]
            for i in customers for j in nodes if i != j
        ) <= capacity

        for i in customers:
            for j in customers:
                if i != j:
                    problem += u[i, k] - u[j, k] + capacity * x[i, j, k] <= capacity - demands[j]

    # A time limit keeps scalability experiments bounded while retaining the best feasible incumbent.
    solver = pulp.HiGHS(timeLimit=time_limit_sec, msg=False)
    if not solver.available():
        raise RuntimeError('HiGHS is unavailable. Install it with: %pip install "pulp>=4,<5" highspy')
    stats = problem.solve(solver)
    execution_time = time.perf_counter() - start_time
    # Read native HiGHS diagnostics because PuLP's generic status omits the bound and MIP gap.
    solver_info = problem.solverModel.getInfo() if hasattr(problem, "solverModel") else None

    def finite_solver_value(attribute):
        value = getattr(solver_info, attribute, None) if solver_info is not None else None
        return float(value) if value is not None and np.isfinite(value) else None

    best_bound = finite_solver_value("mip_dual_bound")
    mip_gap = finite_solver_value("mip_gap")
    explored_nodes = finite_solver_value("mip_node_count")

    if not stats.has_solution:
        return {
            "status": stats.status_str,
            "objective": float("inf"),
            "routes": [],
            "runtime": execution_time,
            "is_optimal": False,
            "best_bound": best_bound,
            "mip_gap": mip_gap,
            "explored_nodes": explored_nodes,
        }

    # Reconstruct ordered depot-to-depot routes from selected binary arcs.
    routes = []
    for k in vehicles:
        route = [0]
        current = 0
        seen = set()
        while True:
            next_node = next(
                (
                    j for j in nodes
                    if j != current
                    and pulp.value(x[current, j, k]) is not None
                    and pulp.value(x[current, j, k]) > 0.5
                ),
                None,
            )
            if next_node is None or next_node == 0:
                if len(route) > 1:
                    route.append(0)
                break
            if next_node in seen:
                raise RuntimeError("Unexpected repeated customer while reconstructing an ILP route.")
            seen.add(next_node)
            route.append(next_node)
            current = next_node
        if len(route) > 2:
            routes.append(route)

    return {
        "status": stats.status_str,
        "objective": float(pulp.value(problem.objective)),
        "routes": routes,
        "runtime": execution_time,
        "is_optimal": stats.status == pulp.LpSolveStatus.Optimal,
        "best_bound": best_bound,
        "mip_gap": mip_gap,
        "explored_nodes": explored_nodes,
    }


exact_results = solve_vrp_exact_ilp(data, time_limit_sec=60)
exact_results


{'status': 'Optimal',
 'objective': 370.559013389067,
 'routes': [[0, 5, 3, 6, 0], [0, 1, 10, 2, 4, 7, 0], [0, 9, 8, 0]],
 'runtime': 7.027132399999573,
 'is_optimal': True,
 'best_bound': 370.55901338906705,
 'mip_gap': 0.0,
 'explored_nodes': 3033.0}

## Method 2 Genetic Algorithm

The GA represents a candidate as a permutation of customer identifiers. A deterministic greedy decoder scans the permutation and starts a new route whenever adding the next customer would exceed vehicle capacity. Tournament selection favours fitter individuals, ordered crossover preserves permutation validity, and swap mutation introduces diversity. A large penalty discourages chromosomes requiring more than the available vehicles.

The GA was selected because it can quickly search a large combinatorial space without enumerating every route. It does not prove optimality and its result varies with the random seed, so repeated runs and convergence history are evaluated.

In [10]:
# Represent each GA individual as a permutation of customer identifiers.
class GeneticAlgorithmVRP:
    def __init__(
        self,
        data,
        pop_size=100,
        generations=200,
        crossover_rate=0.8,
        mutation_rate=0.2,
        seed=42,
    ):
        validate_instance_data(data)
        if pop_size < 2 or generations < 1:
            raise ValueError("GA population size must be at least 2 and generations must be positive.")
        if not 0 <= crossover_rate <= 1 or not 0 <= mutation_rate <= 1:
            raise ValueError("GA crossover and mutation rates must be between 0 and 1.")
        self.data = data
        self.pop_size = pop_size
        self.generations = generations
        self.crossover_rate = crossover_rate
        self.mutation_rate = mutation_rate
        self.num_customers = data["num_customers"]
        self.customers = list(range(1, self.num_customers + 1))
        self.rng = random.Random(seed)

    # Greedily split a permutation into capacity-feasible depot-to-depot routes.
    def decode_chromosome(self, chromosome):
        routes = []
        current_route = [0]
        current_load = 0
        for customer in chromosome:
            demand = int(self.data["demands"][customer])
            if current_load + demand <= self.data["vehicle_capacity"]:
                current_route.append(customer)
                current_load += demand
            else:
                current_route.append(0)
                routes.append(current_route)
                current_route = [0, customer]
                current_load = demand
        current_route.append(0)
        routes.append(current_route)
        return routes

    def calculate_total_distance(self, routes):
        return sum(
            float(self.data["dist_matrix"][a][b])
            for route in routes for a, b in zip(route, route[1:])
        )

    # Penalize solutions that require more routes than the available fleet.
    def fitness(self, chromosome):
        routes = self.decode_chromosome(chromosome)
        distance = self.calculate_total_distance(routes)
        excess_routes = max(0, len(routes) - self.data["num_vehicles"])
        penalty = excess_routes * 1_000_000
        return 1.0 / (distance + penalty + 1e-9)

    # Ordered crossover preserves a valid customer permutation without duplicates.
    def ordered_crossover(self, parent1, parent2):
        if self.rng.random() > self.crossover_rate:
            return parent1.copy(), parent2.copy()
        size = len(parent1)
        a, b = sorted(self.rng.sample(range(size), 2))

        def ox(first, second):
            child = [-1] * size
            child[a:b + 1] = first[a:b + 1]
            remaining = [item for item in second if item not in child]
            cursor = 0
            for position in range(size):
                if child[position] == -1:
                    child[position] = remaining[cursor]
                    cursor += 1
            return child

        return ox(parent1, parent2), ox(parent2, parent1)

    # Swap mutation introduces diversity while keeping the chromosome valid.
    def mutate_swap(self, chromosome):
        if self.rng.random() < self.mutation_rate:
            i, j = self.rng.sample(range(len(chromosome)), 2)
            chromosome[i], chromosome[j] = chromosome[j], chromosome[i]
        return chromosome

    # Evolve the population and retain the best feasible solution found so far.
    def solve(self):
        start_time = time.perf_counter()
        population = [self.rng.sample(self.customers, len(self.customers)) for _ in range(self.pop_size)]
        best_chromosome = None
        best_distance = float("inf")
        history = []

        for _ in range(self.generations):
            fitness_scores = [self.fitness(individual) for individual in population]
            for individual in population:
                routes = self.decode_chromosome(individual)
                distance = self.calculate_total_distance(routes)
                if len(routes) <= self.data["num_vehicles"] and distance < best_distance:
                    best_distance = distance
                    best_chromosome = individual.copy()
            history.append(best_distance)

            # Binary tournament selection favours fitter individuals without removing diversity completely.
            selected = []
            for _ in range(self.pop_size):
                first, second = self.rng.sample(range(self.pop_size), 2)
                winner = population[first] if fitness_scores[first] > fitness_scores[second] else population[second]
                selected.append(winner.copy())

            next_population = []
            for index in range(0, self.pop_size, 2):
                parent1 = selected[index]
                parent2 = selected[(index + 1) % self.pop_size]
                child1, child2 = self.ordered_crossover(parent1, parent2)
                next_population.extend([self.mutate_swap(child1), self.mutate_swap(child2)])
            population = next_population[: self.pop_size]

        runtime = time.perf_counter() - start_time
        if best_chromosome is None:
            return {
                "status": "No feasible solution found",
                "objective": float("inf"),
                "routes": [],
                "runtime": runtime,
                "convergence_history": history,
                "is_optimal": False,
            }
        return {
            "status": "Feasible heuristic",
            "objective": best_distance,
            "routes": self.decode_chromosome(best_chromosome),
            "runtime": runtime,
            "convergence_history": history,
            "is_optimal": False,
        }


ga_solver = GeneticAlgorithmVRP(
    data,
    pop_size=100,
    generations=250,
    crossover_rate=0.85,
    mutation_rate=0.15,
    seed=SEED,
)
ga_results = ga_solver.solve()
ga_results


{'status': 'Feasible heuristic',
 'objective': 382.05014572147445,
 'routes': [[0, 3, 6, 7, 4, 2, 10, 0], [0, 1, 9, 8, 0], [0, 5, 0]],
 'runtime': 0.2736967000000732,
 'convergence_history': [480.88831273051943,
  456.5284859373888,
  456.5284859373888,
  456.5284859373888,
  441.7979307573666,
  413.2339960550214,
  413.2339960550214,
  413.2339960550214,
  413.2339960550214,
  413.2339960550214,
  413.2339960550214,
  383.8798566852661,
  383.8798566852661,
  383.8798566852661,
  383.8798566852661,
  383.8798566852661,
  383.8798566852661,
  383.8798566852661,
  383.8798566852661,
  383.8798566852661,
  383.8798566852661,
  383.8798566852661,
  383.8798566852661,
  382.05014572147445,
  382.05014572147445,
  382.05014572147445,
  382.05014572147445,
  382.05014572147445,
  382.05014572147445,
  382.05014572147445,
  382.05014572147445,
  382.05014572147445,
  382.05014572147445,
  382.05014572147445,
  382.05014572147445,
  382.05014572147445,
  382.05014572147445,
  382.050145721474

## Feasibility validation and baseline comparison

A low objective value is meaningful only if all constraints are satisfied. The validator below independently checks depot returns, customer coverage, duplicates, node indices, route count, and capacity. This guards against comparing an infeasible heuristic route with an ILP solution.


In [11]:
# Independently validate routes rather than trusting either solver's reported objective.
def validate_solution(data, routes):
    errors = []
    customers = []
    route_loads = []
    route_distances = []

    if len(routes) > data["num_vehicles"]:
        errors.append("The number of routes exceeds the vehicle limit.")

    for route_number, route in enumerate(routes, start=1):
        if len(route) < 3 or route[0] != 0 or route[-1] != 0:
            errors.append(f"Route {route_number} does not start and end at the depot.")
        inner = route[1:-1]
        if any(not isinstance(node, (int, np.integer)) or node < 1 or node > data["num_customers"] for node in inner):
            errors.append(f"Route {route_number} contains an invalid customer index.")
        load = int(sum(data["demands"][node] for node in inner))
        if load > data["vehicle_capacity"]:
            errors.append(f"Route {route_number} exceeds capacity: {load} > {data['vehicle_capacity']}.")
        route_loads.append(load)
        route_distances.append(sum(float(data["dist_matrix"][a][b]) for a, b in zip(route, route[1:])))
        customers.extend(inner)

    expected = list(range(1, data["num_customers"] + 1))
    if sorted(customers) != expected:
        errors.append("Customers are missing, duplicated, or visited more than once.")

    return {
        "feasible": not errors,
        "errors": errors,
        "route_loads": route_loads,
        "route_distances": route_distances,
        "total_distance": float(sum(route_distances)),
    }


# Require both feasibility and agreement between the reported and reconstructed objectives.
def assert_result_consistency(data, method, result):
    validation = validate_solution(data, result["routes"])
    if not validation["feasible"]:
        raise ValueError(f"{method} returned an infeasible solution: {validation['errors']}")
    if not np.isclose(validation["total_distance"], result["objective"], rtol=1e-7, atol=1e-7):
        raise ValueError(
            f"{method} objective {result['objective']} does not match reconstructed route distance "
            f"{validation['total_distance']}."
        )
    return validation


# Convert nested route lists into an auditable vehicle-level comparison table.
def build_route_table(data, results_by_method, node_names=None):
    rows = []
    for method, result in results_by_method.items():
        validation = assert_result_consistency(data, method, result)
        for route_number, (route, load, distance) in enumerate(
            zip(result["routes"], validation["route_loads"], validation["route_distances"]),
            start=1,
        ):
            route_labels = [node_names[node] if node_names else str(node) for node in route]
            rows.append(
                {
                    "Method": method,
                    "Vehicle": route_number,
                    "Route": " -> ".join(route_labels),
                    "Load": load,
                    "Capacity": data["vehicle_capacity"],
                    "Capacity_Utilization_pct": 100 * load / data["vehicle_capacity"],
                    "Route_Distance": distance,
                    "Total_Method_Distance": result["objective"],
                    "Runtime_s": result["runtime"],
                    "Feasible": validation["feasible"],
                }
            )
    return pd.DataFrame(rows)


# Apply the same independent checks to the exact and heuristic baseline solutions.
baseline_rows = []
for method, result in (("ILP", exact_results), ("GA", ga_results)):
    validation = assert_result_consistency(data, method, result)
    baseline_rows.append(
        {
            "Method": method,
            "Status": result["status"],
            "Distance": result["objective"],
            "Runtime_s": result["runtime"],
            "Routes_used": len(result["routes"]),
            "Route_loads": validation["route_loads"],
            "Feasible": validation["feasible"],
            "Proven_optimal": result.get("is_optimal", False),
        }
    )

baseline_comparison = pd.DataFrame(baseline_rows)
baseline_gap = 100 * (ga_results["objective"] - exact_results["objective"]) / exact_results["objective"]
display(baseline_comparison)
baseline_route_table = build_route_table(data, {"ILP": exact_results, "GA": ga_results})
display(baseline_route_table.round(4))
print(f"GA distance difference from the ILP incumbent: {baseline_gap:.2f}%")


,Method,Status,Distance,Runtime_s,Routes_used,Route_loads,Feasible,Proven_optimal
0,ILP,Optimal,370.559013,7.027132,3,"[12, 23, 14]",True,True
1,GA,Feasible heuristic,382.050146,0.273697,3,"[23, 20, 6]",True,False


,Method,Vehicle,Route,Load,Capacity,Capacity_Utilization_pct,Route_Distance,Total_Method_Distance,Runtime_s,Feasible
0,ILP,1,0 -> 5 -> 3 -> 6 -> 0,12,25,48.0,162.8309,370.5590,7.0271,True
1,ILP,2,0 -> 1 -> 10 -> 2 -> 4 -> 7 -> 0,23,25,92.0,116.1477,370.5590,7.0271,True
2,ILP,3,0 -> 9 -> 8 -> 0,14,25,56.0,91.5804,370.5590,7.0271,True
3,GA,1,0 -> 3 -> 6 -> 7 -> 4 -> 2 -> 10 -> 0,23,25,92.0,187.8723,382.0501,0.2737,True
4,GA,2,0 -> 1 -> 9 -> 8 -> 0,20,25,80.0,119.1415,382.0501,0.2737,True
5,GA,3,0 -> 5 -> 0,6,25,24.0,75.0363,382.0501,0.2737,True


GA distance difference from the ILP incumbent: 3.10%


## Evaluation 


In [ ]:
# Compare scalability across problem sizes, independent instance seeds, and repeated GA runs.
def run_benchmark_experiments(
    customer_scales=(8, 12, 15, 18),
    instance_seeds=(100, 101),
    ga_repetitions=5,
    ilp_time_limit=10,
):
    instance_rows = []
    ga_run_rows = []
    for n in customer_scales:
        # Multiple instance seeds reduce the risk of drawing conclusions from one favourable layout.
        for instance_seed in instance_seeds:
            print(f"Benchmarking N={n} customers, instance seed={instance_seed}")
            bench_data = generate_vrp_instance(
                num_customers=n,
                vehicle_capacity=30,
                num_vehicles=4,
                seed=instance_seed,
            )
            # Verify the ILP incumbent even when the solver reaches its time limit.
            ilp_result = solve_vrp_exact_ilp(bench_data, time_limit_sec=ilp_time_limit)
            ilp_validation = assert_result_consistency(bench_data, f"ILP N={n} seed={instance_seed}", ilp_result)

            ga_runs = []
            # Repeat the stochastic GA with recorded seeds to quantify variability.
            for repetition in range(ga_repetitions):
                ga_seed = instance_seed * 100 + repetition
                ga = GeneticAlgorithmVRP(
                    bench_data,
                    pop_size=100,
                    generations=250,
                    crossover_rate=0.85,
                    mutation_rate=0.15,
                    seed=ga_seed,
                )
                result = ga.solve()
                validation = assert_result_consistency(
                    bench_data, f"GA N={n} instance={instance_seed} run={repetition + 1}", result
                )
                best_generation = int(np.argmin(result["convergence_history"])) + 1
                ga_runs.append(result)
                ga_run_rows.append(
                    {
                        "N_Customers": n,
                        "Instance_Seed": instance_seed,
                        "GA_Run": repetition + 1,
                        "GA_Seed": ga_seed,
                        "Objective": result["objective"],
                        "Runtime_s": result["runtime"],
                        "Routes_Used": len(result["routes"]),
                        "Feasible": validation["feasible"],
                        "Objective_Verified": np.isclose(validation["total_distance"], result["objective"]),
                        "Best_Found_Generation": best_generation,
                    }
                )

            ga_costs = np.array([result["objective"] for result in ga_runs], dtype=float)
            ga_times = np.array([result["runtime"] for result in ga_runs], dtype=float)
            best_ga = ga_runs[int(np.argmin(ga_costs))]
            proven_gap = (
                100 * (best_ga["objective"] - ilp_result["objective"]) / ilp_result["objective"]
                if ilp_result.get("is_optimal") and ilp_result["objective"] > 0
                else np.nan
            )
            instance_rows.append(
                {
                    "N_Customers": n,
                    "Instance_Seed": instance_seed,
                    "Total_Demand": int(bench_data["demands"].sum()),
                    "ILP_Status": ilp_result["status"],
                    "ILP_Proven_Optimal": ilp_result.get("is_optimal", False),
                    "ILP_Feasible": ilp_validation["feasible"],
                    "ILP_Objective_Verified": np.isclose(ilp_validation["total_distance"], ilp_result["objective"]),
                    "ILP_Cost": ilp_result["objective"],
                    "ILP_Best_Bound": ilp_result.get("best_bound"),
                    "ILP_MIP_Gap": ilp_result.get("mip_gap"),
                    "ILP_Explored_Nodes": ilp_result.get("explored_nodes"),
                    "ILP_Runtime_s": ilp_result["runtime"],
                    "GA_Best_Cost": float(ga_costs.min()),
                    "GA_Mean_Cost": float(ga_costs.mean()),
                    "GA_Cost_Std": float(ga_costs.std(ddof=1)),
                    "GA_Mean_Runtime_s": float(ga_times.mean()),
                    "GA_Runtime_Std_s": float(ga_times.std(ddof=1)),
                    "GA_Feasible_Runs": len(ga_runs),
                    "GA_Repetitions": ga_repetitions,
                    "Proven_Optimality_Gap_pct": proven_gap,
                }
            )

    instance_results = pd.DataFrame(instance_rows)
    ga_run_results = pd.DataFrame(ga_run_rows)
    # Aggregate instance-level outcomes only after retaining all detailed run records.
    summary_rows = []
    for n, group in instance_results.groupby("N_Customers", sort=True):
        ga_group = ga_run_results[ga_run_results["N_Customers"] == n]
        valid_gaps = group["Proven_Optimality_Gap_pct"].dropna()
        summary_rows.append(
            {
                "N_Customers": n,
                "Instances": len(group),
                "Total_Demand": group["Total_Demand"].mean(),
                "ILP_Status": f"{int(group['ILP_Proven_Optimal'].sum())}/{len(group)} optimal",
                "ILP_Proven_Optimal": bool(group["ILP_Proven_Optimal"].all()),
                "ILP_Feasible": bool(group["ILP_Feasible"].all()),
                "ILP_Cost": group["ILP_Cost"].mean(),
                "ILP_Best_Bound": group["ILP_Best_Bound"].mean(),
                "ILP_MIP_Gap": group["ILP_MIP_Gap"].mean(),
                "ILP_Runtime_s": group["ILP_Runtime_s"].mean(),
                "GA_Best_Cost": group["GA_Best_Cost"].mean(),
                "GA_Mean_Cost": ga_group["Objective"].mean(),
                "GA_Cost_Std": ga_group["Objective"].std(ddof=1),
                "GA_Mean_Runtime_s": ga_group["Runtime_s"].mean(),
                "GA_Runtime_Std_s": ga_group["Runtime_s"].std(ddof=1),
                "GA_Feasible_Runs": int(ga_group["Feasible"].sum()),
                "GA_Repetitions": len(ga_group),
                "Proven_Optimality_Gap_pct": valid_gaps.mean() if not valid_gaps.empty else np.nan,
            }
        )
    return pd.DataFrame(summary_rows), instance_results, ga_run_results


df_comparison, df_instance_results, df_ga_runs = run_benchmark_experiments()
print("Per-instance benchmark results")
display(df_instance_results.round(4))
print("Individual GA runs")
display(df_ga_runs.round(4))
print("Scalability summary across instance seeds")
display(df_comparison.round(4))
